# Dataset cleaning pipeline for 3 synthetic code datasets

This notebook:

1. Mounts Google Drive
2. Loads three datasets from Drive
3. Applies a **syntax validity check** to the code response column
4. Applies a **HumanEval leakage check**
5. Saves cleaned datasets, rejected rows, and summary statistics back to Drive

## Notes
- **Safe execution is disabled by default** in this notebook to keep the process lightweight.
- That means this notebook checks **parse validity**, not full runtime validity.
- You can later add an execution step if you want a stricter filter.


In [ ]:
# Install dependencies (run once per Colab session)
!pip -q install datasets scikit-learn pandas tqdm

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Configuration

Edit the paths below to match your Google Drive structure.

The notebook supports:
- `.jsonl`
- `.json`
- `.csv`
- `.parquet`

For each dataset, specify:
- `input_path`
- output directory will be created automatically
- candidate text columns for instruction/prompt
- candidate text columns for code response


In [ ]:
from pathlib import Path

# ----------------------------
# EDIT THESE PATHS
# ----------------------------
DATASETS = {
    "self_instruct": {
        "input_path": "/content/drive/MyDrive/masters_thesis/data/self_instruct_dataset.jsonl",
    },
    "oss_instruct": {
        "input_path": "/content/drive/MyDrive/masters_thesis/data/oss_instruct_dataset.jsonl",
    },
    "complexity_evolved": {
        "input_path": "/content/drive/MyDrive/masters_thesis/data/complexity_evolved_dataset.jsonl",
    },
}

OUTPUT_ROOT = Path("/content/drive/MyDrive/masters_thesis/cleaned_datasets")

# Candidate columns for prompt/instruction text
PROMPT_CANDIDATES = [
    "instruction", "prompt", "question", "task", "problem", "input"
]

# Candidate columns for code/completion text
RESPONSE_CANDIDATES = [
    "response", "output", "completion", "answer", "solution", "generated_code", "code"
]

# Leakage thresholds
PROMPT_SIM_THRESHOLD = 0.80
CODE_SIM_THRESHOLD = 0.92

# Toggle: safe execution check (disabled by default)
RUN_EXECUTION_CHECK = False
EXECUTION_TIMEOUT_SECONDS = 2

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Output root:", OUTPUT_ROOT)

In [ ]:
import ast
import json
import re
import subprocess
import tempfile
import os
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tqdm.pandas()

## Utility functions

In [ ]:
def load_any_dataset(path: str) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    elif suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        if isinstance(obj, list):
            return pd.DataFrame(obj)
        elif isinstance(obj, dict):
            # common fallback for wrapped data formats
            for key in ["data", "records", "items", "examples"]:
                if key in obj and isinstance(obj[key], list):
                    return pd.DataFrame(obj[key])
            return pd.DataFrame([obj])
        else:
            raise ValueError(f"Unsupported JSON structure in {path}")
    elif suffix == ".csv":
        return pd.read_csv(path)
    elif suffix == ".parquet":
        return pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


def save_any_dataset(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix.lower()

    if suffix == ".jsonl":
        df.to_json(path, orient="records", lines=True, force_ascii=False)
    elif suffix == ".csv":
        df.to_csv(path, index=False)
    elif suffix == ".parquet":
        df.to_parquet(path, index=False)
    else:
        raise ValueError(f"Unsupported output type: {suffix}")


def pick_first_existing_column(df: pd.DataFrame, candidates: List[str], label: str) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(
        f"Could not find a {label} column. Available columns: {list(df.columns)}"
    )


def normalize_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_code(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def is_syntax_valid(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except Exception:
        return False


def run_code_safely(code: str, timeout: int = 2) -> Tuple[bool, str]:
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".py", mode="w", encoding="utf-8") as f:
            f.write(code)
            temp_path = f.name

        result = subprocess.run(
            ["python3", temp_path],
            capture_output=True,
            text=True,
            timeout=timeout
        )

        if result.returncode == 0:
            return True, ""
        return False, (result.stderr or "non-zero exit code").strip()

    except subprocess.TimeoutExpired:
        return False, "timeout"
    except Exception as e:
        return False, str(e)
    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

## Load HumanEval reference data

In [ ]:
# The Hugging Face HumanEval dataset card shows the canonical dataset as openai/openai_humaneval.
# We use that here as the leakage reference set.
humaneval = load_dataset("openai/openai_humaneval")["test"]
humaneval_df = humaneval.to_pandas()

humaneval_df["prompt_norm"] = humaneval_df["prompt"].map(normalize_text)
humaneval_df["solution_norm"] = humaneval_df["canonical_solution"].map(normalize_code)
humaneval_df["entry_point_norm"] = humaneval_df["entry_point"].map(normalize_text)

print("HumanEval rows:", len(humaneval_df))
humaneval_df.head(3)

## Leakage detection helpers

In [ ]:
def build_prompt_similarity_model(sample_prompts: List[str], ref_prompts: List[str]):
    all_texts = ref_prompts + sample_prompts
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    vectorizer.fit(all_texts)
    ref_matrix = vectorizer.transform(ref_prompts)
    sample_matrix = vectorizer.transform(sample_prompts)
    sims = cosine_similarity(sample_matrix, ref_matrix)
    return sims


def build_code_similarity_model(sample_codes: List[str], ref_codes: List[str]):
    # Character n-grams work better than word n-grams for code similarity
    all_texts = ref_codes + sample_codes
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
    vectorizer.fit(all_texts)
    ref_matrix = vectorizer.transform(ref_codes)
    sample_matrix = vectorizer.transform(sample_codes)
    sims = cosine_similarity(sample_matrix, ref_matrix)
    return sims


def detect_leakage(df: pd.DataFrame, prompt_col: str, response_col: str) -> pd.DataFrame:
    out = df.copy()

    out["prompt_norm"] = out[prompt_col].map(normalize_text)
    out["response_norm"] = out[response_col].map(normalize_code)

    humaneval_prompt_set = set(humaneval_df["prompt_norm"])
    humaneval_entry_set = set(humaneval_df["entry_point_norm"])
    humaneval_solution_set = set(humaneval_df["solution_norm"])

    out["prompt_exact_match"] = out["prompt_norm"].isin(humaneval_prompt_set)
    out["entry_point_exact_match"] = out["prompt_norm"].isin(humaneval_entry_set)
    out["solution_exact_match"] = out["response_norm"].isin(humaneval_solution_set)

    prompt_sims = build_prompt_similarity_model(
        out["prompt_norm"].fillna("").tolist(),
        humaneval_df["prompt_norm"].fillna("").tolist(),
    )
    out["max_prompt_similarity"] = prompt_sims.max(axis=1)

    code_sims = build_code_similarity_model(
        out["response_norm"].fillna("").tolist(),
        humaneval_df["solution_norm"].fillna("").tolist(),
    )
    out["max_code_similarity"] = code_sims.max(axis=1)

    out["leakage_flag"] = (
        out["prompt_exact_match"]
        | out["entry_point_exact_match"]
        | out["solution_exact_match"]
        | (out["max_prompt_similarity"] >= PROMPT_SIM_THRESHOLD)
        | (out["max_code_similarity"] >= CODE_SIM_THRESHOLD)
    )

    return out

## Cleaning pipeline

In [ ]:
def clean_dataset(
    dataset_name: str,
    dataset_cfg: Dict[str, str],
    output_ext: str = ".jsonl",
) -> Dict[str, object]:
    input_path = dataset_cfg["input_path"]
    print(f"\n=== Processing: {dataset_name} ===")
    print("Input:", input_path)

    df = load_any_dataset(input_path)
    print("Rows loaded:", len(df))

    prompt_col = pick_first_existing_column(df, PROMPT_CANDIDATES, "prompt/instruction")
    response_col = pick_first_existing_column(df, RESPONSE_CANDIDATES, "response/code")

    print("Prompt column:", prompt_col)
    print("Response column:", response_col)

    working = df.copy()
    working["_row_id"] = range(len(working))

    # 1) Syntax validity
    working["syntax_valid"] = working[response_col].fillna("").progress_apply(is_syntax_valid)

    syntax_rejects = working[~working["syntax_valid"]].copy()
    syntax_pass = working[working["syntax_valid"]].copy()

    # 2) Optional execution validity
    if RUN_EXECUTION_CHECK:
        exec_results = syntax_pass[response_col].fillna("").progress_apply(
            lambda x: run_code_safely(x, timeout=EXECUTION_TIMEOUT_SECONDS)
        )
        syntax_pass["exec_valid"] = exec_results.map(lambda x: x[0])
        syntax_pass["exec_error"] = exec_results.map(lambda x: x[1])

        exec_rejects = syntax_pass[~syntax_pass["exec_valid"]].copy()
        exec_pass = syntax_pass[syntax_pass["exec_valid"]].copy()
    else:
        syntax_pass["exec_valid"] = True
        syntax_pass["exec_error"] = ""
        exec_rejects = syntax_pass.iloc[0:0].copy()
        exec_pass = syntax_pass.copy()

    # 3) Leakage detection
    leakage_checked = detect_leakage(exec_pass, prompt_col=prompt_col, response_col=response_col)

    leakage_rejects = leakage_checked[leakage_checked["leakage_flag"]].copy()
    cleaned = leakage_checked[~leakage_checked["leakage_flag"]].copy()

    # Save outputs
    out_dir = OUTPUT_ROOT / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    cleaned_path = out_dir / f"{dataset_name}_cleaned{output_ext}"
    syntax_rejects_path = out_dir / f"{dataset_name}_syntax_rejects{output_ext}"
    exec_rejects_path = out_dir / f"{dataset_name}_execution_rejects{output_ext}"
    leakage_rejects_path = out_dir / f"{dataset_name}_leakage_rejects{output_ext}"
    summary_path = out_dir / f"{dataset_name}_summary.csv"

    save_any_dataset(cleaned, cleaned_path)
    save_any_dataset(syntax_rejects, syntax_rejects_path)
    save_any_dataset(exec_rejects, exec_rejects_path)
    save_any_dataset(leakage_rejects, leakage_rejects_path)

    summary = pd.DataFrame([{
        "dataset_name": dataset_name,
        "input_rows": len(df),
        "syntax_valid_rows": len(syntax_pass),
        "syntax_reject_rows": len(syntax_rejects),
        "execution_valid_rows": len(exec_pass),
        "execution_reject_rows": len(exec_rejects),
        "leakage_reject_rows": len(leakage_rejects),
        "cleaned_rows": len(cleaned),
        "prompt_column_used": prompt_col,
        "response_column_used": response_col,
        "prompt_similarity_threshold": PROMPT_SIM_THRESHOLD,
        "code_similarity_threshold": CODE_SIM_THRESHOLD,
        "run_execution_check": RUN_EXECUTION_CHECK,
    }])
    summary.to_csv(summary_path, index=False)

    print("Saved cleaned dataset to:", cleaned_path)
    print("Saved summary to:", summary_path)

    return {
        "dataset_name": dataset_name,
        "cleaned_path": str(cleaned_path),
        "syntax_rejects_path": str(syntax_rejects_path),
        "exec_rejects_path": str(exec_rejects_path),
        "leakage_rejects_path": str(leakage_rejects_path),
        "summary_path": str(summary_path),
        "summary_df": summary,
    }

## Run all three datasets

In [ ]:
results = []

for dataset_name, dataset_cfg in DATASETS.items():
    result = clean_dataset(dataset_name, dataset_cfg, output_ext=".jsonl")
    results.append(result)

overall_summary = pd.concat([r["summary_df"] for r in results], ignore_index=True)
overall_summary_path = OUTPUT_ROOT / "overall_cleaning_summary.csv"
overall_summary.to_csv(overall_summary_path, index=False)

print("\nAll done.")
print("Overall summary saved to:", overall_summary_path)
overall_summary

## Optional: inspect rows flagged for leakage

In [ ]:
example_dataset = "self_instruct"  # change as needed
example_path = OUTPUT_ROOT / example_dataset / f"{example_dataset}_leakage_rejects.jsonl"

if example_path.exists():
    leakage_df = pd.read_json(example_path, lines=True)
    print("Leakage rows:", len(leakage_df))
    display_cols = [c for c in [
        "instruction", "prompt", "response", "output",
        "max_prompt_similarity", "max_code_similarity",
        "prompt_exact_match", "entry_point_exact_match",
        "solution_exact_match", "leakage_flag"
    ] if c in leakage_df.columns]
    display(leakage_df[display_cols].head(10))
else:
    print("No leakage file found for", example_dataset)

## Interpretation

Because execution is turned off by default, the cleaned datasets are filtered for:

- **syntactic validity**
- **HumanEval contamination / leakage risk**

They are **not guaranteed** to execute correctly at runtime.

That is often a reasonable compromise if:
- you have large datasets,
- compute is limited,
- and your goal is to remove clearly broken syntax plus benchmark contamination before training.

If you later want stricter filtering, set:

```python
RUN_EXECUTION_CHECK = True
```
